In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_curve
)

In [ ]:
df = pd.read_csv(
    "../data/processed/classification_dataset.csv"
)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["late_delivery"].value_counts()

In [ ]:
df["late_delivery"].value_counts(
    normalize=True
) * 100

In [ ]:
class_counts = df["late_delivery"].value_counts().sort_index()

plt.figure(figsize=(7, 5))

class_counts.plot(
    kind="bar"
)

plt.title("Late Delivery Class Distribution")
plt.xlabel("Late Delivery")
plt.ylabel("Number of Orders")

plt.xticks(
    [0, 1],
    ["On Time", "Late"],
    rotation=0
)

plt.show()

In [ ]:
X = df.drop(
    columns=["late_delivery"]
)

y = df["late_delivery"]

In [ ]:
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
id_columns = [
    "order_id",
    "customer_id"
]

X = X.drop(
    columns=id_columns,
    errors="ignore"
)

In [ ]:
leakage_columns = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "delivery_delay_days",
    "delivery_days"
]

X = X.drop(
    columns=leakage_columns,
    errors="ignore"
)

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

X = X.drop(
    columns=date_columns,
    errors="ignore"
)

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [ ]:
print("Numeric Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(
    "Train distribution:"
)

print(
    y_train.value_counts(
        normalize=True
    ) * 100
)

print()

print(
    "Test distribution:"
)

print(
    y_test.value_counts(
        normalize=True
    ) * 100
)

In [ ]:
majority_class = y_train.mode()[0]

baseline_prediction = np.full(
    len(y_test),
    majority_class
)

In [ ]:
baseline_accuracy = accuracy_score(
    y_test,
    baseline_prediction
)

baseline_precision = precision_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_prediction,
    zero_division=0
)

In [ ]:
print("Baseline Accuracy :", baseline_accuracy)
print("Baseline Precision:", baseline_precision)
print("Baseline Recall   :", baseline_recall)
print("Baseline F1       :", baseline_f1)

In [ ]:
logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
logistic_model.fit(
    X_train,
    y_train
)

In [ ]:
logistic_pred = logistic_model.predict(
    X_test
)

In [ ]:
logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
print(
    "Accuracy:",
    accuracy_score(
        y_test,
        logistic_pred
    )
)

print(
    "Precision:",
    precision_score(
        y_test,
        logistic_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        logistic_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        logistic_pred,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        logistic_prob
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    logistic_pred
)

print(cm)

In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "On Time",
        "Late"
    ]
)

disp.plot()

plt.title(
    "Confusion Matrix - Logistic Regression"
)

plt.show()

In [ ]:
rf_classifier = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                max_depth=15,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]
)

In [ ]:
rf_classifier.fit(
    X_train,
    y_train
)

In [ ]:
rf_pred = rf_classifier.predict(
    X_test
)

In [ ]:
rf_prob = rf_classifier.predict_proba(
    X_test
)[:, 1]

In [ ]:
print(
    "Accuracy:",
    accuracy_score(
        y_test,
        rf_pred
    )
)

print(
    "Precision:",
    precision_score(
        y_test,
        rf_pred,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        rf_pred,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        rf_pred,
        zero_division=0
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        rf_prob
    )
)

In [ ]:
print(
    classification_report(
        y_test,
        logistic_pred,
        target_names=[
            "On Time",
            "Late"
        ]
    )
)

In [ ]:
print(
    classification_report(
        y_test,
        rf_pred,
        target_names=[
            "On Time",
            "Late"
        ]
    )
)

In [ ]:
classification_results = pd.DataFrame([
    {
        "Model": "Baseline",
        "Accuracy": accuracy_score(
            y_test,
            baseline_prediction
        ),
        "Precision": precision_score(
            y_test,
            baseline_prediction,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            baseline_prediction,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            baseline_prediction,
            zero_division=0
        ),
        "ROC-AUC": np.nan
    },

    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(
            y_test,
            logistic_pred
        ),
        "Precision": precision_score(
            y_test,
            logistic_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            logistic_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            logistic_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            logistic_prob
        )
    },

    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(
            y_test,
            rf_pred
        ),
        "Precision": precision_score(
            y_test,
            rf_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            rf_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_test,
            rf_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            rf_prob
        )
    }
])

In [ ]:
classification_results

In [ ]:
cm_rf = confusion_matrix(
    y_test,
    rf_pred
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=[
        "On Time",
        "Late"
    ]
)

disp.plot()

plt.title(
    "Confusion Matrix - Random Forest"
)

plt.show()

In [ ]:
logistic_fpr, logistic_tpr, _ = roc_curve(
    y_test,
    logistic_prob
)

rf_fpr, rf_tpr, _ = roc_curve(
    y_test,
    rf_prob
)

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(
    logistic_fpr,
    logistic_tpr,
    label="Logistic Regression"
)

plt.plot(
    rf_fpr,
    rf_tpr,
    label="Random Forest"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
rf_estimator = rf_classifier.named_steps[
    "model"
]

In [ ]:
rf_preprocessor = rf_classifier.named_steps[
    "preprocessor"
]

In [ ]:
feature_names = (
    rf_preprocessor
    .get_feature_names_out()
)

In [ ]:
importances = rf_estimator.feature_importances_

In [ ]:
feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
})

In [ ]:
feature_importance = (
    feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
)

In [ ]:
feature_importance.head(20)

In [ ]:
top_features = (
    feature_importance
    .head(20)
    .sort_values(
        "importance"
    )
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_features["feature"],
    top_features["importance"]
)

plt.xlabel("Importance")
plt.ylabel("Feature")

plt.title(
    "Top Features for Late Delivery Prediction"
)

plt.show()

In [ ]:
import joblib

In [ ]:
best_classifier = rf_classifier

In [ ]:
joblib.dump(
    best_classifier,
    "../models/late_delivery_model.pkl"
)

In [ ]:
loaded_classifier = joblib.load(
    "../models/late_delivery_model.pkl"
)

In [ ]:
loaded_pred = loaded_classifier.predict(
    X_test.head(10)
)

loaded_pred

In [ ]:
loaded_prob = loaded_classifier.predict_proba(
    X_test.head(10)
)[:, 1]

loaded_prob